In [14]:
# Precomputation:

from collections import deque

class SokobanHeuristic:
    def __init__(self, board_matrix, goals):
        """
        board_matrix: Ma trận bản đồ (chứa tường '%', ô trống ' ')
        goals: Danh sách tọa độ các ô đích [(r1, c1), (r2, c2), ...]
        """
        self.goals = goals
        self.board = board_matrix
        # Dictionary lưu khoảng cách: self.maze_dist[goal_pos][tile_pos] = distance
        self.maze_dist = {}

        # Xây dựng bảng tra cứu khoảng cách cho từng ô đích
        for goal in self.goals:
            self.maze_dist[goal] = self._bfs_from_goal(goal)

    def _bfs_from_goal(self, start_goal):
        """
        Chạy BFS từ 1 ô đích ra toàn bộ bản đồ để tính khoảng cách ngắn nhất qua tường '%'
        """
        distances = {start_goal: 0}
        queue = deque([start_goal])

        # 4 hướng di chuyển: Trên, Dưới, Trái, Phải
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        while queue:
            r, c = queue.popleft()
            current_dist = distances[(r, c)]

            for dr, dc in directions:
                nr, nc = r + dr, c + dc

                # Điều kiện di chuyển: Không ra ngoài biên và không phải là tường '%'
                if self.board[nr][nc] != '%' and (nr, nc) not in distances:
                    distances[(nr, nc)] = current_dist + 1
                    queue.append((nr, nc))

        return distances

    def evaluate(self, boxes_positions):
        total_h = 0
        for box in boxes_positions:
            min_dist = float('inf')
            for goal in self.goals:
                if box in self.maze_dist[goal]:
                    dist = self.maze_dist[goal][box]
                    if dist < min_dist:
                        min_dist = dist

            if min_dist == float('inf'):
                return float('inf')  # Deadlock

            total_h += min_dist

        return total_h

In [9]:
# h(S) calculator for A*

def evaluate(self, boxes_positions):
        """
        boxes_positions: Danh sách/Tuple tọa độ các thùng ở trạng thái hiện tại [(rb1, cb1), ...]
        """
        total_h = 0

        for box in boxes_positions:
            # Nếu thùng đã nằm đúng ô đích, chi phí đóng góp bằng 0
            # Ngược lại, tìm khoảng cách mê cung nhỏ nhất tới một trong các ô đích
            min_dist_for_box = float('inf')

            for goal in self.goals:
                # Trích xuất khoảng cách đã tính sẵn từ bảng tra cứu
                if box in self.maze_dist[goal]:
                    dist = self.maze_dist[goal][box]
                    if dist < min_dist_for_box:
                        min_dist_for_box = dist

            # Trường hợp thùng bị kẹt ở góc không thể đến bất kỳ đích nào
            if min_dist_for_box == float('inf'):
                return float('inf') # Trạng thái ngõ tóm (Deadlock)

            total_h += min_dist_for_box

        return total_h

In [10]:
# State

import heapq

class SokobanState:
    def __init__(self, agent_pos, boxes):
        self.agent_pos = agent_pos  # Tọa độ (r, c) của người chơi
        self.boxes = frozenset(boxes)  # Tập hợp tọa độ các thùng

    def __eq__(self, other):
        return self.agent_pos == other.agent_pos and self.boxes == other.boxes

    def __hash__(self):
        return hash((self.agent_pos, self.boxes))

    def __lt__(self, other):
        # Hỗ trợ so sánh trong Priority Queue khi chi phí ưu tiên bằng nhau
        return False


class SokobanProblem:
    def __init__(self, board_matrix):
        self.walls = set()
        self.goals = set()
        boxes = set()
        agent_pos = None

        # Đọc ma trận bản đồ để trích xuất các thông tin ban đầu
        for r, row in enumerate(board_matrix):
            for c, val in enumerate(row):
                if val == '%':
                    self.walls.add((r, c))
                elif val == 'A':
                    agent_pos = (r, c)
                elif val == 'B':
                    boxes.add((r, c))
                elif val == 'D':
                    self.goals.add((r, c))
                elif val == 'C':  # Thùng đã nằm trên ô đích
                    boxes.add((r, c))
                    self.goals.add((r, c))

        self.initial_state = SokobanState(agent_pos, boxes)

    def is_goal(self, state):
        # Trạng thái đích: Tất cả tọa độ thùng trùng khớp với tập hợp các ô đích
        return state.boxes == self.goals

    def get_successors(self, state):
        """
        Sinh ra các trạng thái kế tiếp hợp lệ từ trạng thái hiện tại.
        Trả về danh sách dạng: [(state_mới, tên_hành_động, chi_phí_bước_đi)]
        """
        ACTIONS = {
            'North': (-1, 0),
            'South': (1, 0),
            'West': (0, -1),
            'East': (0, 1)
        }
        successors = []
        r, c = state.agent_pos

        for action_name, (dr, dc) in ACTIONS.items():
            nr, nc = r + dr, c + dc  # Vị trí mới của Agent nếu di chuyển

            # 1. Nếu đụng tường -> Bỏ qua
            if (nr, nc) in self.walls:
                continue

            # 2. Nếu đụng thùng -> Kiểm tra xem có đẩy được thùng không
            if (nr, nc) in state.boxes:
                nnr, nnc = nr + dr, nc + dc  # Vị trí mới của thùng sau khi đẩy

                # Nếu đằng sau thùng là tường hoặc một thùng khác -> Bỏ qua
                if (nnr, nnc) in self.walls or (nnr, nnc) in state.boxes:
                    continue

                # Đẩy thùng thành công
                new_boxes = set(state.boxes)
                new_boxes.remove((nr, nc))
                new_boxes.add((nnr, nnc))
                new_state = SokobanState((nr, nc), new_boxes)
                successors.append((new_state, action_name, 1))

            # 3. Di chuyển vào ô trống
            else:
                new_state = SokobanState((nr, nc), state.boxes)
                successors.append((new_state, action_name, 1))

        return successors

In [11]:
# UCS:

def ucs_search(problem):
    start_state = problem.initial_state

    # Priority Queue lưu phần tử dạng: (g_cost, state, path_actions)
    frontier = []
    heapq.heappush(frontier, (0, start_state, []))

    # Bảng lưu chi phí g(n) nhỏ nhất đến mỗi trạng thái
    visited = {start_state: 0}
    nodes_expanded = 0

    while frontier:
        g_cost, state, path = heapq.heappop(frontier)

        # Kiểm tra trạng thái đích
        if problem.is_goal(state):
            return path, g_cost, nodes_expanded

        nodes_expanded += 1

        for next_state, action, step_cost in problem.get_successors(state):
            new_g = g_cost + step_cost

            # Cập nhật nếu tìm thấy đường đi ngắn hơn tới next_state
            if next_state not in visited or new_g < visited[next_state]:
                visited[next_state] = new_g
                heapq.heappush(frontier, (new_g, next_state, path + [action]))

    return None, float('inf'), nodes_expanded  # Không tìm thấy đường đi

In [12]:
# A*:

def a_star_search(problem, heuristic_fn):
    start_state = problem.initial_state
    h_start = heuristic_fn.evaluate(start_state.boxes)

    # Priority Queue lưu phần tử dạng: (f_score, g_cost, state, path_actions)
    frontier = []
    heapq.heappush(frontier, (0 + h_start, 0, start_state, []))

    # Bảng lưu chi phí g(n) nhỏ nhất tới mỗi trạng thái
    visited = {start_state: 0}
    nodes_expanded = 0

    while frontier:
        f_score, g_cost, state, path = heapq.heappop(frontier)

        # Kiểm tra trạng thái đích
        if problem.is_goal(state):
            return path, g_cost, nodes_expanded

        nodes_expanded += 1

        for next_state, action, step_cost in problem.get_successors(state):
            new_g = g_cost + step_cost

            if next_state not in visited or new_g < visited[next_state]:
                visited[next_state] = new_g

                # Tính giá trị f(n) = g(n) + h(n)
                h_cost = heuristic_fn.evaluate(next_state.boxes)

                # Nếu h_cost = inf tức là trạng thái rơi vào Deadlock (kẹt thùng)
                if h_cost != float('inf'):
                    new_f = new_g + h_cost
                    heapq.heappush(frontier, (new_f, new_g, next_state, path + [action]))

    return None, float('inf'), nodes_expanded  # Không tìm thấy đường đi

In [22]:
# Main

import time
import os

def read_map(file_path):
    """
    Đọc ma trận bản đồ từ file .txt
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.rstrip('\r\n') for line in f.readlines()]
    return lines


def run_experiment(map_file):
    print(f"\n==================================================")
    print(f"   ĐANG THỬ NGHIỆM MAP: {map_file}")
    print(f"==================================================")

    board_matrix = read_map(map_file)
    problem = SokobanProblem(board_matrix)
    heuristic = SokobanHeuristic(board_matrix, problem.goals)

    # 1. Chạy thuật toán UCS
    start_time = time.perf_counter()
    path_ucs, cost_ucs, nodes_ucs = ucs_search(problem)
    time_ucs = (time.perf_counter() - start_time) * 1000  # Đổi sang ms

    # 2. Chạy thuật toán A*
    start_time = time.perf_counter()
    path_a_star, cost_a_star, nodes_a_star = a_star_search(problem, heuristic)
    time_a_star = (time.perf_counter() - start_time) * 1000  # Đổi sang ms

    # In kết quả so sánh
    print(f"{'Tiêu chí':<25} | {'UCS':<20} | {'A* Search':<20}")
    print("-" * 70)
    print(f"{'Tổng chi phí (Total Cost)':<25} | {cost_ucs:<20} | {cost_a_star:<20}")
    print(f"{'Số bước hành động':<25} | {len(path_ucs) if path_ucs else 0:<20} | {len(path_a_star) if path_a_star else 0:<20}")
    print(f"{'Số Node đã mở (Expanded)':<25} | {nodes_ucs:<20} | {nodes_a_star:<20}")
    print(f"{'Thời gian chạy (ms)':<25} | {time_ucs:<20.2f} | {time_a_star:<20.2f}")

    if path_a_star:
        print(f"\nDanh sách hành động (A*): {path_a_star}")


if __name__ == "__main__":
    # Danh sách các file map cần kiểm thử
    map_files = ["map_small.txt", "map_medium.txt", "map_large.txt"]

    for map_file in map_files:
        if os.path.exists(map_file):
           run_experiment(map_file)
        else:
            print(f"Lỗi: Không tìm thấy file '{map_file}' trong thư mục!")


   ĐANG THỬ NGHIỆM MAP: map_small.txt
Tiêu chí                  | UCS                  | A* Search           
----------------------------------------------------------------------
Tổng chi phí (Total Cost) | 6                    | 6                   
Số bước hành động         | 6                    | 6                   
Số Node đã mở (Expanded)  | 113                  | 46                  
Thời gian chạy (ms)       | 1.36                 | 0.78                

Danh sách hành động (A*): ['East', 'South', 'West', 'South', 'East', 'East']

   ĐANG THỬ NGHIỆM MAP: map_medium.txt
Tiêu chí                  | UCS                  | A* Search           
----------------------------------------------------------------------
Tổng chi phí (Total Cost) | 11                   | 11                  
Số bước hành động         | 11                   | 11                  
Số Node đã mở (Expanded)  | 2387                 | 239                 
Thời gian chạy (ms)       | 20.60                | 2.